# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities in the dataset are referenced by their `@id`.

In [ ]:
# List available record sets and their fields by @id
record_sets = dataset.record_sets

print("Available record sets:")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field name: {field.name}, @id: {field.id}, dataType: {field.data_type}")

# (Optional) Show example records for a specific record set by @id
if record_sets:
    selected_record_set_id = record_sets[0].id
    print(f"\nExample records from record set '@id': {selected_record_set_id}")
    for rec in dataset.records(record_set=selected_record_set_id):
        print(rec)
        break  # Display only one example record

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis.

Use the record set and field `@id` from the overview.

In [ ]:
# Extract data from each record set
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

# Load records into pandas DataFrames, using the record set @id as key
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show columns for primary record set
primary_record_set_id = record_set_ids[0] if record_set_ids else None
if primary_record_set_id:
    print(f"Columns in record set '@id': {primary_record_set_id}")
    print(dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Entities are referenced by `@id`.

In [ ]:
# Choose the primary record set for EDA
record_set_id = primary_record_set_id
df = dataframes[record_set_id]

# List numeric fields and their @id
numeric_fields = [field.id for field in dataset.record_set(record_set_id).fields if field.data_type in ['schema:Integer', 'schema:Float', 'schema:Number']]

print("Numeric fields in the selected record set:")
for nf in numeric_fields:
    print(nf)

# Choose GAD-7 Score field for filtering and normalization
# (Replace with actual field @id from dataset; example: 'gad_7_score', but always use @id)
gad7_field_id = None
for field in dataset.record_set(record_set_id).fields:
    if 'gad' in field.name.lower() and 'score' in field.name.lower():
        gad7_field_id = field.id
        break

if gad7_field_id and gad7_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[gad7_field_id] > threshold]
    print(f"Filtered records with {gad7_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize GAD-7 scores
    filtered_df[f"{gad7_field_id}_normalized"] = (filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()) / filtered_df[gad7_field_id].std()

    print(f"Normalized {gad7_field_id} for filtered records:")
    display(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

    # Group by a demographic field (e.g., gender) using the @id
    gender_field_id = None
    for field in dataset.record_set(record_set_id).fields:
        if 'gender' in field.name.lower():
            gender_field_id = field.id
            break
    if gender_field_id and gender_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(gender_field_id)[gad7_field_id].mean().reset_index()
        print(f"Grouped data by {gender_field_id} (GAD-7 average by gender):")
        display(grouped_df)
else:
    print("No GAD-7 score field found by @id.")

## 5. Visualization
Visualize data distributions and relationships between fields in the dataset.

This example plots the distribution of GAD-7 scores and average score by gender, referencing all columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if gad7_field_id and gad7_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[gad7_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of GAD-7 Scores ({gad7_field_id})")
    plt.xlabel("GAD-7 Score")
    plt.ylabel("Count")
    plt.show()

    if gender_field_id and gender_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[gender_field_id], y=df[gad7_field_id])
        plt.title(f"GAD-7 Scores by Gender ({gender_field_id})")
        plt.xlabel("Gender")
        plt.ylabel("GAD-7 Score")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides valuable insights into mental health indicators using standardized assessment scores (GAD-7, PHQ-9, PSQ) among Kilifi County residents.
- Exploratory analysis reveals differences in survey responses across demographic categories such as gender.
- The use of Croissant `@id` enables consistent referencing and future-proof processing of each field and record set.

Further analysis can build upon this notebook, including modeling, deeper subgroup analysis, or application of AI tools using `mlcroissant` data structures.